In [0]:
-- ============================================================
-- SILVER LAYER
-- Cleaning and standardizing EPA emissions data
-- ============================================================

CREATE OR REPLACE TABLE emissions.silver.emissions_clean
USING DELTA
AS

SELECT
    -- Geographic identifiers
    LPAD(CAST(state_id AS STRING), 2, '0') AS state_id,
    UPPER(TRIM(state_abbr)) AS state_abbr,

    LPAD(CAST(county_id AS STRING), 5, '0') AS county_id,
    TRIM(county_name) AS county_name,
    TRIM(county_state_name) AS county_state_name,

    -- Geographic coordinates
    TRY_CAST(latitude AS DOUBLE) AS latitude,
    TRY_CAST(longitude AS DOUBLE) AS longitude,

    -- Population
    TRY_CAST(
        REPLACE(CAST(population AS STRING), ',', '')
        AS BIGINT
    ) AS population,

    -- Greenhouse gas emissions
    TRY_CAST(
        REPLACE(
            CAST(`GHG emissions mtons CO2e` AS STRING),
            ',',
            ''
        )
        AS DOUBLE
    ) AS ghg_emissions_mtons_co2e,

    -- Metadata inherited from Bronze
    ingestion_timestamp,
    source_file

FROM emissions.bronze.emissions_raw;


--------------------------------------------------------

SELECT COUNT(*) AS silver_row_count
FROM emissions.silver.emissions_clean;

-----------------------------------------

SELECT *
FROM emissions.silver.emissions_clean
LIMIT 10;

------------------------------------------------

DESCRIBE emissions.silver.emissions_clean;

--------------------------------------------------
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN population IS NULL THEN 1 ELSE 0 END)
        AS null_population,

    SUM(CASE WHEN ghg_emissions_mtons_co2e IS NULL THEN 1 ELSE 0 END)
        AS null_emissions,

    SUM(CASE WHEN latitude IS NULL THEN 1 ELSE 0 END)
        AS null_latitude,

    SUM(CASE WHEN longitude IS NULL THEN 1 ELSE 0 END)
        AS null_longitude

FROM emissions.silver.emissions_clean;

--------------------------------------------------------------

SELECT
    (SELECT COUNT(*)
     FROM emissions.bronze.emissions_raw) AS bronze_rows,

    (SELECT COUNT(*)
     FROM emissions.silver.emissions_clean) AS silver_rows;